In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d
import gtsam

/home/gautham/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
## Load images from a directory
def load_data(image_dir):
    Images = []
    file_list = sorted(os.listdir(image_dir))
    for file_name in file_list:
        if file_name.lower().endswith(('.png')):
            img_path = os.path.join(image_dir, file_name)
            img = cv2.imread(img_path)
            if img is not None:
                Images.append(img)
            else:
                print(f"Warning: Failed to read {img_path}")
    return Images

image_dir = 'buddha_images/'
Images = load_data(image_dir)
num_images = len(Images)
print(f"Loaded {num_images} images from {image_dir}")

Loaded 24 images from buddha_images/


In [3]:
def normalize_points(pts):
    if pts is None or len(pts) == 0:
        return None, None 

    centroid = np.mean(pts, axis=0)
    pts_centered = pts - centroid
    avg_dist = np.mean(np.sqrt(np.sum(pts_centered**2, axis=1)))

    if avg_dist < 1e-6:
        return None, None 

    scale = np.sqrt(2) / avg_dist
    T = np.array([[scale, 0, -scale * centroid[0]],
                  [0, scale, -scale * centroid[1]],
                  [0, 0, 1]])

    pts_h = np.hstack((pts, np.ones((pts.shape[0], 1))))
    pts_normalized_h = (T @ pts_h.T).T

    return pts_normalized_h[:, :2], T

In [4]:
## Estimate the Fundamental Matrix using the normalized 8-point algorithm
def estimate_fundamental_matrix(pts1, pts2):
    pts1_normalized, T1 = normalize_points(pts1)
    if pts1_normalized is None:
        return None, None, None
    pts2_normalized, T2 = normalize_points(pts2)
    if pts2_normalized is None:
        return None, None, None
    
    A = np.zeros((pts1_normalized.shape[0], 9))
    for i in range(pts1_normalized.shape[0]):
        x1, y1 = pts1_normalized[i]
        x2, y2 = pts2_normalized[i]
        A[i] = [x2*x1, x2*y1, x2, y2*x1, y2*y1, y2, x1, y1, 1]
        
    # Solve Af = 0 using SVD
    U, S, Vt = np.linalg.svd(A)
    F_normalized = Vt[-1].reshape(3, 3)

    # Enforce rank-2 constraint
    U, S, Vt_f = np.linalg.svd(F_normalized)
    S[2] = 0
    F_normalized_rank2 = U @ np.diag(S) @ Vt_f
    # Denormalize
    F = T2.T @ F_normalized_rank2 @ T1
    return F / F[2, 2], T1, T2

In [5]:
## Estimate F Matrix using RANSAC with Sampson's distance
def ransac_fundamental_matrix(pts1, pts2, num_iterations=5000, threshold=1.0):
    max_inliers = []
    best_F = None

    for _ in range(num_iterations):
        idx = np.random.choice(len(pts1), 8, replace=False)
        F_candidate, _, _ = estimate_fundamental_matrix(pts1[idx], pts2[idx])
        if F_candidate is None:
            continue

        # Compute Sampson distance
        pts1_h = np.hstack((pts1, np.ones((pts1.shape[0], 1))))
        pts2_h = np.hstack((pts2, np.ones((pts2.shape[0], 1))))
        
        Fx1 = F_candidate @ pts1_h.T
        Ftx2 = F_candidate.T @ pts2_h.T
        
        denom = Fx1[0]**2 + Fx1[1]**2 + Ftx2[0]**2 + Ftx2[1]**2
        err = (np.sum(pts2_h * (F_candidate @ pts1_h.T).T, axis=1))**2 / denom
        
        inliers = np.where(err < threshold)[0]
        
        if len(inliers) > len(max_inliers):
            max_inliers = inliers
            best_F = F_candidate

    return best_F, max_inliers

In [6]:
def load_all_intrinsics(file_path):
    intrinsics_dict = {}
    
    with open(file_path, 'r') as f:
        for line in f:
            # Skip comments and empty lines
            if not line.strip() or line.startswith('#'):
                continue
            
            params = line.split()
            
            # Extract parameters based on SIMPLE_RADIAL model
            # Format: CAMERA_ID, MODEL, WIDTH, HEIGHT, f, cx, cy, k1
            cam_id = int(params[0])
            f_val = float(params[4])
            cx = float(params[5])
            cy = float(params[6])
            k1 = float(params[7])
            
            # Construct the 3x3 K matrix
            K = np.array([[f_val, 0,    cx],
                          [0,    f_val, cy],
                          [0,    0,    1.0]], dtype=float)
            
            # Print the matrix for this specific camera
            print(f"Camera ID: {cam_id}")
            print(f"Focal Length: {f_val}")
            print(f"Principal Point: ({cx}, {cy})")
            print(f"Radial Distortion (k1): {k1}")
            print(f"Intrinsic Matrix K:\n{K}\n")
            
            # Store in dictionary for use in the pipeline
            intrinsics_dict[cam_id] = {
                'K': K,
                'k1': k1
            }
            
    return intrinsics_dict

camera_txt_path = "cameras.txt"
all_intrinsics = load_all_intrinsics(camera_txt_path)

Camera ID: 1
Focal Length: 1700.5960173760209
Principal Point: (540.0, 960.0)
Radial Distortion (k1): 0.04460722023943839
Intrinsic Matrix K:
[[1.70059602e+03 0.00000000e+00 5.40000000e+02]
 [0.00000000e+00 1.70059602e+03 9.60000000e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]

Camera ID: 2
Focal Length: 1696.9933605815868
Principal Point: (540.0, 960.0)
Radial Distortion (k1): 0.04684414515917654
Intrinsic Matrix K:
[[1.69699336e+03 0.00000000e+00 5.40000000e+02]
 [0.00000000e+00 1.69699336e+03 9.60000000e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]

Camera ID: 3
Focal Length: 1707.438317595852
Principal Point: (540.0, 960.0)
Radial Distortion (k1): 0.04551770441526508
Intrinsic Matrix K:
[[1.70743832e+03 0.00000000e+00 5.40000000e+02]
 [0.00000000e+00 1.70743832e+03 9.60000000e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]

Camera ID: 4
Focal Length: 1707.3555769968862
Principal Point: (540.0, 960.0)
Radial Distortion (k1): 0.04351000987799287
Intrinsic Matri

In [7]:
def compute_essential_matrix(F, K_i, K_j):
    # Essential matrix formula: E = Kj^T * F * Ki
    E = K_j.T @ F @ K_i
    
    # Enforce internal constraint (SVD cleanup)
    U, S, Vt = np.linalg.svd(E)
    S = [1, 1, 0]
    E_corrected = U @ np.diag(S) @ Vt
    
    return E_corrected

In [50]:
## Feature extraction and matching using SIFT
def extract_features_and_matches(input_images):
    sift = cv2.SIFT_create(nfeatures=5000)
    bf = cv2.BFMatcher()
    keypoints_list = []
    descriptors_list = []
    matches_list = {}

    for img in input_images:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        keypoints, descriptors = sift.detectAndCompute(gray, None)
        keypoints_list.append(keypoints)
        descriptors_list.append(descriptors)

    for i in range(num_images):
        for j in range(i + 1, num_images):
            if descriptors_list[i] is None or descriptors_list[j] is None:
                continue

            matches = bf.knnMatch(descriptors_list[i], descriptors_list[j], k=2)
            good_matches = [m for m, n in matches if m.distance < 0.75 * n.distance]
            
            if len(good_matches) >= 15:
                pts1 = np.array([keypoints_list[i][m.queryIdx].pt for m in good_matches])
                pts2 = np.array([keypoints_list[j][m.trainIdx].pt for m in good_matches])

                F, inliers = ransac_fundamental_matrix(pts1, pts2)
                if F is not None and len(inliers) > 20:
                    matches_list[(i,j)] = [good_matches[idx] for idx in inliers]
                    print(f"Verified pair ({i}, {j}) with {len(inliers)} inliers.")

    print(f"{len(matches_list)} matches found.")
    return keypoints_list, descriptors_list, matches_list

keypoints, descriptors, matches = extract_features_and_matches(Images)

Verified pair (0, 1) with 304 inliers.
Verified pair (0, 2) with 167 inliers.
Verified pair (0, 3) with 140 inliers.
Verified pair (0, 4) with 46 inliers.
Verified pair (0, 5) with 43 inliers.
Verified pair (0, 6) with 24 inliers.
Verified pair (0, 7) with 32 inliers.
Verified pair (0, 8) with 21 inliers.
Verified pair (0, 10) with 27 inliers.
Verified pair (0, 11) with 62 inliers.
Verified pair (0, 18) with 22 inliers.
Verified pair (0, 19) with 29 inliers.
Verified pair (0, 20) with 24 inliers.
Verified pair (0, 21) with 23 inliers.
Verified pair (0, 22) with 33 inliers.
Verified pair (1, 2) with 202 inliers.
Verified pair (1, 3) with 160 inliers.
Verified pair (1, 4) with 57 inliers.
Verified pair (1, 5) with 42 inliers.
Verified pair (1, 6) with 30 inliers.
Verified pair (2, 3) with 197 inliers.
Verified pair (2, 4) with 57 inliers.
Verified pair (2, 5) with 61 inliers.
Verified pair (2, 6) with 22 inliers.
Verified pair (2, 7) with 23 inliers.
Verified pair (2, 11) with 27 inliers

In [51]:
def build_global_tracks(num_images, keypoints, matches):
    """
    Implements the Unordered Feature Tracking algorithm using Union-Find.
   
    """
    # Create a singleton for each feature (MakeSet)
    # We map (image_id, kp_index) to a unique integer ID for Union-Find
    node_to_id = {}
    id_to_node = []
    for i in range(num_images):
        for k_idx in range(len(keypoints[i])):
            node_to_id[(i, k_idx)] = len(id_to_node)
            id_to_node.append((i, k_idx))

    num_nodes = len(id_to_node)
    parent = np.arange(num_nodes)
    rank = np.zeros(num_nodes, dtype=int) # Helps keep the tree shallow

    def find(i):
        # Find with Path Compression
        if parent[i] == i:
            return i
        parent[i] = find(parent[i])
        return parent[i]

    def union(i, j):
        # Union by Rank
        root_i = find(i)
        root_j = find(j)
        if root_i != root_j:
            if rank[root_i] < rank[root_j]:
                parent[root_i] = root_j
            elif rank[root_i] > rank[root_j]:
                parent[root_j] = root_i
            else:
                parent[root_j] = root_i
                rank[root_i] += 1

    # For each pairwise match do join (Union)
    for (i, j), matches in matches.items():
        for m in matches:
            id_i = node_to_id[(i, m.queryIdx)]
            id_j = node_to_id[(j, m.trainIdx)]
            union(id_i, id_j)

    # Return each connected set as a track
    tracks = {}
    for node_idx in range(num_nodes):
        root = find(node_idx)
        if root not in tracks:
            tracks[root] = []
        tracks[root].append(id_to_node[node_idx])

    # Filter: Keep only tracks seen in 2 or more images
    final_tracks = {tid: obs for tid, obs in tracks.items() if len(obs) >= 2}

    # Initialize a list of dictionaries for each image in your dataset
    img_kp_to_track = [{} for _ in range(len(keypoints))]

    # Fill the lookup table: img_kp_to_track[image_id][kp_index] = global_track_id
    # 'tracks' is the variable returned by your build_global_tracks function
    for track_id, observations in tracks.items():
        for (img_id, kp_idx) in observations:
            img_kp_to_track[img_id][kp_idx] = track_id

    print(f"Lookup table created. Ready to map {len(tracks)} global tracks.")
    
    print(f"Merged {len(matches)} pairs into {len(final_tracks)} global tracks.")
    return final_tracks, img_kp_to_track

# Usage with your existing variables:
tracks, img_kp_to_track = build_global_tracks(num_images, keypoints, matches)

Lookup table created. Ready to map 12477 global tracks.
Merged 83 pairs into 1657 global tracks.


In [52]:
## Estimate camera pose from Essential Matrix
def estimate_camera_pose(E):
    U, S, Vt = np.linalg.svd(E)
    if np.linalg.det(U) < 0:
        U[:, -1] *= -1
    if np.linalg.det(Vt) < 0:
        Vt[-1, :] *= -1

    W = np.array([[0, -1, 0],
                  [1, 0, 0],
                  [0, 0, 1]])

    R1 = U @ W @ Vt
    R2 = U @ W.T @ Vt
    t = U[:, 2]
    return (R1, t), (R1, -t), (R2, t), (R2, -t)

In [53]:
def triangulate_points(P1, P2, pts1, pts2):
    # OpenCV requires 2xN arrays of float type
    pts1_input = pts1.T.astype(np.float32)
    pts2_input = pts2.T.astype(np.float32)
    
    # Triangulate
    points_4d = cv2.triangulatePoints(P1, P2, pts1_input, pts2_input)
    
    # Convert Homogeneous (4D) to Euclidean (3D)
    # Divide x,y,z by w
    points_3d = (points_4d[:3] / points_4d[3]).T
    
    return points_3d

In [54]:
def cheirality_check(poses, K_i, K_j, pts1, pts2):
    """
    Finds the valid camera pose by ensuring points are in front of both cameras.
    Updated to support unique intrinsic matrices for each viewpoint.
    """
    # Define the first camera projection matrix (at the origin)
    P1 = K_i @ np.hstack((np.eye(3), np.zeros((3, 1))))
    
    max_positive_depth = -1
    best_pose = None

    for R, t in poses:
        # Define the second camera projection matrix
        P2 = K_j @ np.hstack((R, t.reshape(3, 1)))
        
        # Triangulate points in 3D using the specific P matrices
        points_3d = triangulate_points(P1, P2, pts1, pts2)

        # Check depth in Camera 1: The Z-coordinate in Camera 1's frame
        depth1 = points_3d[:, 2]
        
        # Check depth in Camera 2: Transform points to Camera 2's frame
        # P_cam2 = R * P_world + t
        points_cam2 = (R @ points_3d.T + t.reshape(3, 1))
        depth2 = points_cam2[2, :]

        # Count points that have positive depth in BOTH cameras
        positive_depth_count = np.sum((depth1 > 0) & (depth2 > 0))

        if positive_depth_count > max_positive_depth:
            max_positive_depth = positive_depth_count
            best_pose = (R, t)
    
    print(f"Cheirality Check: Found valid pose with {max_positive_depth} points in front of cameras.")
    return best_pose

In [55]:
def calculate_median_parallax(pts1, pts2, K_i, K_j, R, t):
    """
    Calculates the median angle between rays from two cameras to 3D points.
   
    """
    # Define Projection Matrices
    # Camera i is at Origin [I|0], Camera j is at [R|t]
    P1 = K_i @ np.hstack((np.eye(3), np.zeros((3, 1))))
    P2 = K_j @ np.hstack((R, t.reshape(3, 1)))
    
    # Triangulate Points in 3D
    pts3d = triangulate_points(P1, P2, pts1, pts2)
    
    # Get Camera Centers
    C1 = np.zeros(3)
    C2 = -R.T @ t # Center of second camera in first camera's frame
    
    angles = []
    for pt in pts3d:
        # Vector from C1 to point and C2 to point
        v1 = pt - C1
        v2 = pt - C2
        
        # Calculate angle between vectors
        cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
        angle = np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0)))
        angles.append(angle)
        
    return np.median(angles)

In [56]:
def find_best_seed(matches, all_intrinsics, keypoints):
    """
    Finds the best starting pair based on Inliers and Median Parallax.
    """
    best_pair = None
    max_inliers = 0
    best_pose_data = None
    
    # Check top candidates (sorted by inlier count)
    sorted_candidates = sorted(matches.keys(), key=lambda k: len(matches[k]), reverse=True)
    
    for (i, j) in sorted_candidates[:15]:
        inlier_matches = matches[(i, j)]
        
        # Get specific K matrices for images i and j
        # Adding 1 because COLMAP IDs typically start at 1
        K_i = all_intrinsics[i+1]['K']
        K_j = all_intrinsics[j+1]['K']
        
        pts1 = np.array([keypoints[i][m.queryIdx].pt for m in inlier_matches])
        pts2 = np.array([keypoints[j][m.trainIdx].pt for m in inlier_matches])

        # Estimate Fundamental Matrix
        F, inliers = ransac_fundamental_matrix(pts1, pts2)
        if F is None:
            continue

        E = compute_essential_matrix(F, K_i, K_j)
            
        # Compute Essential Matrix using both Kj and Ki
        poses = estimate_camera_pose(E)
        
        # Cheirality Check (Finds the valid R and t)
        # Use specific Kj and Ki for projection inside this function
        R_rel, t_rel = cheirality_check(poses, K_i, K_j, pts1, pts2)
        
        if R_rel is None:
            continue

        # Calculate Median Parallax Angle
        parallax = calculate_median_parallax(pts1, pts2, K_i, K_j, R_rel, t_rel)
        
        # Threshold: > 2.0 degrees ensures motion wasn't too small
        if parallax > 2.0:
            print(f"Candidate ({i}, {j}): {len(inlier_matches)} inliers, {parallax:.2f}° parallax - VALID\n")
            if len(inlier_matches) > max_inliers:
                max_inliers = len(inlier_matches)
                best_pair = (i, j)
                best_pose_data = (R_rel, t_rel)
        else:
            print(f"Candidate ({i}, {j}): {parallax:.2f}° parallax - TOO LOW (Narrow Baseline)\n")

    return best_pair, best_pose_data

# Execute the search
best_pair, seed_pose = find_best_seed(matches, all_intrinsics, keypoints)

Cheirality Check: Found valid pose with 290 points in front of cameras.
Candidate (0, 1): 0.07° parallax - TOO LOW (Narrow Baseline)

Cheirality Check: Found valid pose with 199 points in front of cameras.
Candidate (1, 2): 0.14° parallax - TOO LOW (Narrow Baseline)

Cheirality Check: Found valid pose with 192 points in front of cameras.
Candidate (2, 3): 0.42° parallax - TOO LOW (Narrow Baseline)

Cheirality Check: Found valid pose with 167 points in front of cameras.
Candidate (0, 2): 0.49° parallax - TOO LOW (Narrow Baseline)

Cheirality Check: Found valid pose with 160 points in front of cameras.
Candidate (1, 3): 0.82° parallax - TOO LOW (Narrow Baseline)

Cheirality Check: Found valid pose with 144 points in front of cameras.
Candidate (20, 21): 145 inliers, 2.42° parallax - VALID

Cheirality Check: Found valid pose with 140 points in front of cameras.
Candidate (0, 3): 0.70° parallax - TOO LOW (Narrow Baseline)

Cheirality Check: Found valid pose with 129 points in front of came

In [57]:
def plot_3d_map(map_3d_dict, camera_poses_dict, all_intrinsics, Images):
    """
    Plots the 3D point cloud and camera frustums.
    Adapted for Dictionary-based Map and Multi-K Intrinsics.
    """
    geometry_list = []
    
    # 1. Prepare Point Cloud
    points = np.array(list(map_3d_dict.values()))
    if points.size > 0:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points)
        pcd.paint_uniform_color([0.0, 0.0, 0.0]) 
        geometry_list.append(pcd)
    else:
        print("No 3D points in the map to display.")

    # 2. Prepare Camera Frustums
    all_frustum_points = []
    all_lines = []
    all_colors = []
    
    # Calculate scale based on the point cloud size
    scale = 1.0
    if points.size > 0:
        max_dim = np.max(np.linalg.norm(points, axis=1))
        scale = max_dim / 300.0 # Adjust this to change frustum size
    
    h, w = Images[0].shape[:2]
    # Standard corners in 2D
    corners_2d = np.array([[0, 0, 1], [w, 0, 1], [w, h, 1], [0, h, 1]]).T

    for i, (img_idx, (R, t)) in enumerate(camera_poses_dict.items()):
        # Get specific K for this camera to unproject corners correctly
        K = all_intrinsics[img_idx + 1]['K']
        K_inv = np.linalg.inv(K)
        corners_3d = K_inv @ corners_2d
        
        # Camera-to-world transformation
        T_world = np.eye(4)
        T_world[:3, :3] = R.T
        T_world[:3, 3] = (-R.T @ t.reshape(3, 1)).flatten()
        
        C_world = T_world[:3, 3] # Optical Center
        
        # Transform frustum corners to world space
        p1 = (T_world @ np.append(corners_3d[:, 0] * scale, 1))[:3]
        p2 = (T_world @ np.append(corners_3d[:, 1] * scale, 1))[:3]
        p3 = (T_world @ np.append(corners_3d[:, 2] * scale, 1))[:3]
        p4 = (T_world @ np.append(corners_3d[:, 3] * scale, 1))[:3]

        current_base_idx = len(all_frustum_points)
        all_frustum_points.extend([C_world, p1, p2, p3, p4])

        # Define the 8 lines of the frustum pyramid
        lines = [
            [current_base_idx, current_base_idx + 1], [current_base_idx, current_base_idx + 2],
            [current_base_idx, current_base_idx + 3], [current_base_idx, current_base_idx + 4],
            [current_base_idx + 1, current_base_idx + 2], [current_base_idx + 2, current_base_idx + 3],
            [current_base_idx + 3, current_base_idx + 4], [current_base_idx + 4, current_base_idx + 1]
        ]
        all_lines.extend(lines)
        
        # Color the first camera green and others red
        color = [0, 1, 0] if i == 0 else [1, 0, 0]
        all_colors.extend([color] * len(lines))

    if all_frustum_points:
        line_set = o3d.geometry.LineSet(
            points=o3d.utility.Vector3dVector(np.array(all_frustum_points)),
            lines=o3d.utility.Vector2iVector(np.array(all_lines))
        )
        line_set.colors = o3d.utility.Vector3dVector(np.array(all_colors))
        geometry_list.append(line_set)

    # 3. Render
    o3d.visualization.draw_geometries(geometry_list, window_name="Buddha Reconstruction")

# Usage:
# plot_3d_map(map_3d, camera_poses, all_intrinsics, Images)

In [58]:
## Initialize global map and camera poses
map_3d = {}         # Global Track_ID -> np.array([x, y, z])
camera_poses = {}    # Image_ID -> (R, t)

# Define the Seed Pair
seed_i, seed_j = 16, 18

# Extract Inlier Data for the Seed Pair
# matches_list is your dictionary containing verified matches
matches_candidates = matches[(seed_i, seed_j)]
pts1 = np.array([keypoints[seed_i][m.queryIdx].pt for m in matches_candidates])
pts2 = np.array([keypoints[seed_j][m.trainIdx].pt for m in matches_candidates])

# Geometric Verification and Intrinsics
F_seed, inlier_mask = ransac_fundamental_matrix(pts1, pts2)
pts1_inliers = pts1[inlier_mask]
pts2_inliers = pts2[inlier_mask]

# Get unique K matrices for these specific iPhone images
Ki = all_intrinsics[seed_i + 1]['K']
Kj = all_intrinsics[seed_j + 1]['K']

# Recover Initial Pose
E = compute_essential_matrix(F_seed, Ki, Kj) # Uses Ki and Kj
poses = estimate_camera_pose(E)
R_seed, t_seed = cheirality_check(poses, Ki, Kj, pts1_inliers, pts2_inliers)

# Triangulate Initial 3D Points
P1 = Ki @ np.hstack((np.eye(3), np.zeros((3, 1))))
P2 = Kj @ np.hstack((R_seed, t_seed.reshape(3, 1)))
points_3d_raw = triangulate_points(P1, P2, pts1_inliers, pts2_inliers)

# Store in Global Map using Track IDs
# Set first camera at origin and second camera relative to it
camera_poses[seed_i] = (np.eye(3), np.zeros(3))
camera_poses[seed_j] = (R_seed, t_seed)

for idx, pt3d in enumerate(points_3d_raw):
    # Filter: Keep points in front of camera
    if pt3d[2] > 0: 
        # Identify which global track this point belongs to
        match_obj = matches_candidates[inlier_mask[idx]]
        track_id = img_kp_to_track[seed_i][match_obj.queryIdx]
        
        # Add to the global map
        map_3d[track_id] = pt3d

print(f"Initialized Map with {len(map_3d)} 3D points.")
print(f"Seed Pair ({seed_i}, {seed_j}) registered successfully.")

Cheirality Check: Found valid pose with 97 points in front of cameras.
Initialized Map with 96 3D points.
Seed Pair (16, 18) registered successfully.


In [59]:
# After seed pair triangulation
camera_poses[seed_i] = (np.eye(3), np.zeros(3))
camera_poses[seed_j] = (R_seed, t_seed)

for idx, pt3d in enumerate(points_3d_raw):
    # Check depth in camera 1 (world frame, since cam1 is at origin)
    depth_cam1 = pt3d[2]
    
    # Check depth in camera 2
    depth_cam2 = (R_seed @ pt3d + t_seed)[2]
    
    if depth_cam1 > 0.1 and depth_cam2 > 0.1:  # ✅ Check BOTH
        match_obj = matches_candidates[inlier_mask[idx]]
        track_id = img_kp_to_track[seed_i][match_obj.queryIdx]
        map_3d[track_id] = pt3d

bad_points = 0
for idx, pt3d in enumerate(points_3d_raw):
    depth_cam2 = (R_seed @ pt3d + t_seed)[2]
    if pt3d[2] > 0 and depth_cam2 <= 0:
        bad_points += 1
print(f"Points passing world Z but failing camera 2 depth: {bad_points}")

Points passing world Z but failing camera 2 depth: 0


In [60]:
# Setup the processing orders
# Forward: from seed_j+1 to the end
forward_indices = range(seed_j + 1, len(keypoints))
# Reverse: from seed_i-1 back to 0
reverse_indices = range(seed_i - 1, -1, -1)

def run_pnp_mapping(indices, map_3d, camera_poses, img_kp_to_track, matches):
    """
    Standardizes the PnP and Mapping expansion for a given sequence of indices.
   
    """
    for i in indices:
        if i in camera_poses: continue
            
        # PART 1: Find 2D-3D Correspondences
        pts_3d, pts_2d = [], []
        for kp_idx, track_id in img_kp_to_track[i].items():
            if track_id in map_3d:
                pts_3d.append(map_3d[track_id])
                pts_2d.append(keypoints[i][kp_idx].pt)

        # Skip if we don't have enough global tracks already in the map
        if len(pts_3d) < 12: 
            print(f"Skipping image {i}: Only {len(pts_3d)} correspondences.")
            continue

        # PART 2: Solve PnP with Unique Intrinsics
        K_curr = all_intrinsics[i + 1]['K']
        success, rvec, tvec, inliers = cv2.solvePnPRansac(
            np.array(pts_3d, dtype=np.float32), 
            np.array(pts_2d, dtype=np.float32), 
            K_curr, None, flags=cv2.SOLVEPNP_ITERATIVE,
            reprojectionError=8.0
        )

        if success:
            R_curr, _ = cv2.Rodrigues(rvec)
            t_curr = tvec.flatten()
            camera_poses[i] = (R_curr, t_curr)
            
            # PART 3: Map Expansion (Triangulation)
            # To maintain scale consistency, we triangulate against the 'previous' 
            # registered image in the sequence
            P2 = K_curr @ np.hstack((R_curr, t_curr.reshape(3, 1)))
            
            new_points_count = 0
            # Look at a small window of neighbors (e.g., last 3 registered)
            for neighbor_idx in range(max(0, i-3), min(len(keypoints), i+4)):
                if neighbor_idx not in camera_poses or neighbor_idx == i:
                    continue
                
                pair = tuple(sorted((neighbor_idx, i)))
                if pair not in matches: continue
                
                K_p = all_intrinsics[neighbor_idx + 1]['K']
                R_p, t_p = camera_poses[neighbor_idx]
                P1 = K_p @ np.hstack((R_p, t_p.reshape(3, 1)))

                pts1_tri, pts2_tri, tids_tri = [], [], []
                for m in matches[pair]:
                    idx_i = m.queryIdx if pair[0] == i else m.trainIdx
                    idx_p = m.trainIdx if pair[0] == i else m.queryIdx
                    
                    tid = img_kp_to_track[i][idx_i]
                    if tid not in map_3d:
                        pts1_tri.append(keypoints[neighbor_idx][idx_p].pt)
                        pts2_tri.append(keypoints[i][idx_i].pt)
                        tids_tri.append(tid)

                if len(pts1_tri) > 0:
                    new_3d = triangulate_points(P1, P2, np.array(pts1_tri), np.array(pts2_tri))
                    for j, pt3d in enumerate(new_3d):
                        if pt3d[2] > 0: # Check depth
                            map_3d[tids_tri[j]] = pt3d
                            new_points_count += 1
            
            print(f"Registered {i}: {len(inliers)} inliers, added {new_points_count} points.")

# Execute Bi-Directional Pass
print("Starting Forward Pass")
run_pnp_mapping(forward_indices, map_3d, camera_poses, img_kp_to_track, matches)

print("\nStarting Reverse Pass")
run_pnp_mapping(reverse_indices, map_3d, camera_poses, img_kp_to_track, matches)

Starting Forward Pass
Registered 19: 51 inliers, added 73 points.
Registered 20: 83 inliers, added 98 points.
Registered 21: 120 inliers, added 90 points.
Registered 22: 86 inliers, added 55 points.
Registered 23: 125 inliers, added 55 points.

Starting Reverse Pass
Registered 15: 93 inliers, added 62 points.
Registered 14: 79 inliers, added 57 points.
Registered 13: 37 inliers, added 28 points.
Registered 12: 38 inliers, added 34 points.
Registered 11: 64 inliers, added 75 points.
Registered 10: 64 inliers, added 52 points.
Registered 9: 59 inliers, added 45 points.
Registered 8: 62 inliers, added 47 points.
Registered 7: 37 inliers, added 30 points.
Registered 6: 40 inliers, added 70 points.
Registered 5: 58 inliers, added 66 points.
Registered 4: 44 inliers, added 44 points.
Registered 3: 44 inliers, added 49 points.
Registered 2: 82 inliers, added 141 points.
Registered 1: 153 inliers, added 105 points.
Registered 0: 179 inliers, added 140 points.


In [61]:
from collections import defaultdict

# Build obs_by_track first
obs_by_track = defaultdict(list)

for i, kp_map in enumerate(img_kp_to_track):
    if i >= len(keypoints) or keypoints[i] is None:
        continue
    for kp_idx, tid in kp_map.items():
        u, v = keypoints[i][kp_idx].pt
        obs_by_track[tid].append((i, (float(u), float(v))))

# Now run diagnostic
print("\n=== Diagnosing Map Quality ===")
bad_tracks = []
for tid, pt3d in map_3d.items():
    observing_cams = [cam_idx for cam_idx, _ in obs_by_track[tid] if cam_idx in camera_poses]
    
    for cam_idx in observing_cams:
        R_w2c, t_w2c = camera_poses[cam_idx]
        depth = (R_w2c @ pt3d + t_w2c)[2]
        
        if depth <= 0:
            bad_tracks.append((tid, cam_idx, depth))
            break

print(f"Total tracks in map: {len(map_3d)}")
print(f"Tracks with negative depth in at least one observer: {len(bad_tracks)}")

if bad_tracks[:10]:
    print("\nFirst 10 bad tracks (tid, cam_idx, depth):")
    for t in bad_tracks[:10]:
        print(f"  Track {t[0]}: depth {t[2]:.3f} in camera {t[1]}")


=== Diagnosing Map Quality ===
Total tracks in map: 1483
Tracks with negative depth in at least one observer: 5

First 10 bad tracks (tid, cam_idx, depth):
  Track 15112: depth -0.602 in camera 21
  Track 15113: depth -0.602 in camera 21
  Track 2614: depth -0.499 in camera 2
  Track 2622: depth -0.499 in camera 2
  Track 900: depth -1.527 in camera 0


In [62]:
## Visualize final reconstruction (pre-optimized)
plot_3d_map(map_3d, camera_poses, all_intrinsics, Images)

print(f"Total Number of 3D Points: {len(map_3d)}")
print(f"Total Number of Camera Poses: {len(camera_poses)}")

Total Number of 3D Points: 1483
Total Number of Camera Poses: 23


In [65]:
from gtsam.symbol_shorthand import X, L
from collections import defaultdict

# ---------------------------------------------------------
# 1. Pre-build Track Observations (Raw Pixels)
# ---------------------------------------------------------
obs_by_track = defaultdict(list)

for i, kp_map in enumerate(img_kp_to_track):
    if i >= len(keypoints) or keypoints[i] is None:
        continue
    for kp_idx, tid in kp_map.items():
        u, v = keypoints[i][kp_idx].pt
        obs_by_track[tid].append((i, (float(u), float(v))))

# ---------------------------------------------------------
# 2. Filter Observations with Negative Depth
# ---------------------------------------------------------
MIN_DEPTH = 0.1

valid_obs_by_track = defaultdict(list)
filtered_obs_count = 0

for tid, obs in obs_by_track.items():
    if tid not in map_3d:
        continue
    pt3d = np.array(map_3d[tid])
    
    for (cam_idx, uv) in obs:
        if cam_idx not in camera_poses:
            continue
        R_w2c, t_w2c = camera_poses[cam_idx]
        depth = (R_w2c @ pt3d + t_w2c)[2]
        
        if depth > MIN_DEPTH:
            valid_obs_by_track[tid].append((cam_idx, uv))
        else:
            filtered_obs_count += 1

# Only keep tracks with at least 2 valid observations
valid_tracks = {tid for tid, obs in valid_obs_by_track.items() if len(obs) >= 2}
print(f"Filtered {filtered_obs_count} bad observations.")
print(f"Using {len(valid_tracks)}/{len(map_3d)} tracks for BA.")

# ---------------------------------------------------------
# 3. Initialize Graph and Values
# ---------------------------------------------------------
graph = gtsam.NonlinearFactorGraph()
initial_values = gtsam.Values()

measurement_noise = gtsam.noiseModel.Robust.Create(
    gtsam.noiseModel.mEstimator.Huber.Create(1.345),
    gtsam.noiseModel.Isotropic.Sigma(2, 1.0)
)

# ---------------------------------------------------------
# 4. Insert Camera Poses & Single Pose Anchor
# ---------------------------------------------------------
for i, (R_w2c, t_w2c) in camera_poses.items():
    R_c2w = R_w2c.T
    t_c2w = -R_w2c.T @ t_w2c.reshape(3, 1)
    Twc = gtsam.Pose3(gtsam.Rot3(R_c2w), gtsam.Point3(t_c2w.flatten()))
    
    initial_values.insert(X(i), Twc.retract(1e-4 * np.random.randn(6, 1)))

    if i == 10: 
        pose_prior_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([1e-4]*3 + [1e-6]*3))
        graph.add(gtsam.PriorFactorPose3(X(i), Twc, pose_prior_noise))

# ---------------------------------------------------------
# 5. Insert Landmarks and Projection Factors
# ---------------------------------------------------------
num_projections = 0
landmarks_added = set()

for tid in valid_tracks:
    pt3d = np.array(map_3d[tid])
    initial_values.insert(L(tid), gtsam.Point3(*pt3d))
    landmarks_added.add(tid)

    for (cam_idx, (u, v)) in valid_obs_by_track[tid]:
        K_vals = all_intrinsics[cam_idx + 1]['K']
        gtsam_K = gtsam.Cal3_S2(K_vals[0,0], K_vals[1,1], K_vals[0,1], K_vals[0,2], K_vals[1,2])

        # throwCheirality=False allows optimization to continue if points drift behind cameras
        graph.add(gtsam.GenericProjectionFactorCal3_S2(
            gtsam.Point2(u, v), measurement_noise, X(cam_idx), L(tid), gtsam_K, False, True
        ))
        num_projections += 1

print(f"Added {len(landmarks_added)} landmarks with {num_projections} projections.")

# ---------------------------------------------------------
# 6. Optimize and Extract Results
# ---------------------------------------------------------
try:
    params = gtsam.LevenbergMarquardtParams()
    params.setMaxIterations(100)
    params.setRelativeErrorTol(1e-5)
    params.setAbsoluteErrorTol(1e-5)
    
    optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial_values, params)
    result = optimizer.optimize()

    initial_error = graph.error(initial_values)
    final_error = graph.error(result)
    final_rmse = np.sqrt(final_error / (2 * num_projections))
    
    print(f"\nOptimization Complete.")
    print(f"Initial error: {initial_error:.2f}")
    print(f"Final error: {final_error:.2f}")
    print(f"Error reduction: {100*(1 - final_error/initial_error):.1f}%")
    print(f"Final RMSE: {final_rmse:.4f} px")

    # Extract optimized poses (convert back to OpenCV convention)
    optimized_poses = {}
    for i in camera_poses.keys():
        if result.exists(X(i)):
            p = result.atPose3(X(i))
            R_opt = p.rotation().matrix().T
            t_opt = -p.rotation().matrix().T @ p.translation().reshape(3,1)
            optimized_poses[i] = (R_opt, t_opt.flatten())

    # Extract optimized 3D points
    optimized_map_3d = {tid: np.array(result.atPoint3(L(tid))) for tid in landmarks_added if result.exists(L(tid))}

    plot_3d_map(optimized_map_3d, optimized_poses, all_intrinsics, Images)

except RuntimeError as e:
    print(f"BA Failed with error: {e}")

Filtered 12 bad observations.
Using 1477/1483 tracks for BA.
Added 1477 landmarks with 5306 projections.

Optimization Complete.
Initial error: 233164.35
Final error: 172461.75
Error reduction: 26.0%
Final RMSE: 4.0313 px
